# Topic Modeling in Google Colab

This notebook recreates the topic-model workflow from the local Python script, but is set up to run in Google Colab.

Use the setup block to install packages, then choose one data source below: either upload a CSV, mount a Google Drive folder, or pull a Kaggle dataset via `kagglehub`.


In [ ]:
!pip install -q pandas numpy matplotlib gensim nltk scikit-learn kagglehub

print('Packages installed.')


Choose a dataset source

If your CSV is in Google Drive, use the drive option. If it is on Kaggle, the Kaggle option downloads it automatically. If you want to upload from your machine, use the upload option.

For other datasets, a package-based import is also often the easiest option, for example: `kagglehub` for Kaggle or `datasets` for Hugging Face.


In [ ]:
import os
from pathlib import Path

DATA_SOURCE = 'kaggle'  # options: 'drive', 'kaggle', 'upload', 'local'
TEXT_COLUMN = 'headline_text'  # change this for other datasets, e.g. 'abstract'
DATA_PATH = '/content/drive/MyDrive/topic_model/abcnews-date-text.csv'

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f'File not found in Drive: {DATA_PATH}. Update DATA_PATH to match your file path.')

elif DATA_SOURCE == 'kaggle':
    import kagglehub
    dataset_path = kagglehub.dataset_download('therohk/million-headlines')
    DATA_PATH = os.path.join(dataset_path, 'abcnews-date-text.csv')
    print('Kaggle dataset downloaded to:', dataset_path)

elif DATA_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded))

elif DATA_SOURCE == 'local':
    # Example for a file already in the notebook environment
    DATA_PATH = '/content/abcnews-date-text.csv'

print('Data source selected:', DATA_SOURCE)
print('Data path:', DATA_PATH)
print('Exists:', os.path.exists(DATA_PATH))

# Optional example for Hugging Face datasets:
# from datasets import load_dataset
# ds = load_dataset('CShorten/ML-ArXiv-Papers')
# print(ds)


Basic Imports

In [ ]:
import numpy as np
import pandas as pd
import gensim
from gensim import corpora
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models.coherencemodel import CoherenceModel
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import nltk
import matplotlib.pyplot as plt


Settings

In [ ]:

np.random.seed(2026)
nltk.download('wordnet')
nltk.download('omw-1.4')
stemmer = SnowballStemmer('english')

# model settings
NUM_TOPICS = 25
PASSES = 10
WORKERS = 1

# data settings
USE_SUBSET = True
DO_TESTING = True
TRUNCATE_SIZE = 50000
INSPECT_ROW = 4310
UNSEEN_TEXT = 'The stock market is experiencing unprecedented volatility due to global economic uncertainty.'

# dictionary settings
MAX_WORDS = 100000
MIN_WORDS = 15
MAX_PERCENTAGE = 0.5

print('Dependencies loaded.')


Preprocessing functions

In [ ]:
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

def preprocess(text):
    if pd.isna(text) or not isinstance(text, str):
        return []

    result = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
    return result

print('Preprocessing functions ready.')


In [ ]:
# Load data from CSV
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Could not find the dataset at {DATA_PATH}. Check the file path or change DATA_SOURCE.')

data = pd.read_csv(DATA_PATH)
print('Dataset loaded. Shape:', data.shape)
print(data.head())
print('Missing values:\n', data.isna().sum())

if TEXT_COLUMN not in data.columns:
    raise KeyError(f'Text column {TEXT_COLUMN} not found. Available columns: {list(data.columns[:10])}')

data_text = data[[TEXT_COLUMN]].copy()
data_text['index'] = data_text.index

documents = data_text
if USE_SUBSET:
    documents = data_text.truncate(before=0, after=TRUNCATE_SIZE)

print('Documents kept for analysis:', len(documents))
print(documents.head())


In [ ]:
# Inspect a sample row before and after preprocessing
if DO_TESTING:
    print('\nDocument length is', len(documents))
    print(documents.head())

    doc_sample = documents[documents['index'] == INSPECT_ROW].values[0][0]
    print(f'\nOriginal row #{INSPECT_ROW}:')
    print(doc_sample)
    print(f'\nTokenized and lemmatized row #{INSPECT_ROW}:')
    print(preprocess(doc_sample))


In [ ]:
# Preprocess all documents
processed_docs = documents[TEXT_COLUMN].map(preprocess)

if DO_TESTING:
    print('\nFirst 10 processed documents:')
    for doc in processed_docs.head(10):
        print(doc)

# Create the dictionary
dictionary = corpora.Dictionary(processed_docs)

if DO_TESTING:
    print('\nFirst 10 dictionary entries:')
    for i, (token_id, token) in enumerate(dictionary.items()):
        print(token_id, token)
        if i >= 9:
            break

# Filter very rare or overly common terms
dictionary.filter_extremes(no_below=MIN_WORDS, no_above=MAX_PERCENTAGE, keep_n=MAX_WORDS)

bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
print('Dictionary size after filtering:', len(dictionary))
print('Bag-of-words corpus size:', len(bow_corpus))

if DO_TESTING:
    this_row = bow_corpus[INSPECT_ROW] if INSPECT_ROW < len(bow_corpus) else bow_corpus[0]
    print(f'\nExample BOW output for row #{INSPECT_ROW}:')
    for term_id, count in this_row[:10]:
        print(f'Word ID: {term_id} ({dictionary[term_id]}) appears {count} time(s).')


In [ ]:
# Optional coherence sweep: test multiple topic counts
DO_COHERENCE_TEST = False

if DO_COHERENCE_TEST:
    def compute_coherence_values(dictionary, corpus, texts, start=2, limit=40, step=6):
        coherence_values = []
        for num_topics in range(start, limit, step):
            lda_model = gensim.models.LdaMulticore(
                corpus,
                num_topics=num_topics,
                id2word=dictionary,
                passes=10,
                workers=1,
            )
            cm = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
            coherence_values.append(cm.get_coherence())
        return coherence_values

    coherence_values = compute_coherence_values(
        dictionary=dictionary,
        corpus=bow_corpus,
        texts=processed_docs,
        start=10,
        limit=200,
        step=15,
    )

    x = list(range(10, 200, 15))
    plt.plot(x, coherence_values)
    plt.xlabel('Num Topics')
    plt.ylabel('Coherence score')
    plt.title('Topic Coherence by Number of Topics')
    plt.show()
else:
    print('Coherence sweep skipped. Set DO_COHERENCE_TEST = True to run it.')


In [ ]:
# Train the LDA model
try:
    print('Creating LDA model...')
    lda_model = gensim.models.LdaMulticore(
        bow_corpus,
        num_topics=NUM_TOPICS,
        id2word=dictionary,
        passes=PASSES,
        workers=WORKERS,
    )

    print('Model created. Top terms by topic:')
    for idx, topic in lda_model.print_topics(-1):
        print(f'Topic: {idx} | Words: {topic}')
except Exception as e:
    print('Something went wrong while training the model.')
    print(str(e))


In [ ]:
# Assign topics to unseen text
bow_vector = dictionary.doc2bow(preprocess(UNSEEN_TEXT))
print(f'\nUnseen text: {UNSEEN_TEXT}')
print('Top predicted topics:')
for index, score in sorted(lda_model[bow_vector], key=lambda tup: -tup[1]):
    print(f'Score: {score}\t Topic {index}: {lda_model.print_topic(index, 5)}')


## Notes and next steps

- Change `TEXT_COLUMN` to match your dataset if you are using an abstract or description field rather than a headline column.
- For large datasets, start with a subset of rows using `USE_SUBSET = True` and `TRUNCATE_SIZE`.
- The model is interpretive: topic labels depend heavily on preprocessing choices, stopwords, and the number of topics.
- If you want to save the model or export topic assignments: use `lda_model.save(...)` and merge the topic scores back into your DataFrame.
- Next possible improvements: tune the number of topics using coherence scoring, add custom stopwords, and compare lemmatization vs. stemming.
